# MSDSS v4 骨架分組／特徵 v5：離線訓練與獨立評估
上傳新版 dataset.zip。主實驗 train/test/validation=450/150/150，另有96筆保留語境群 challenge。
一般 test 保留未見骨架群（家族可共享）；challenge 保留語境群及其近似變體；Sylius validation 只用固定抽樣的外部輸入。Medusa 排除。
預設 MODE="offline"。資料仍未實機驗證；所有結果標示離線分類實驗。未改寫原文或正規化大小寫／空白／編碼。

同骨架及近似變體整組分配；分析視圖只用於分組，模型仍讀原始輸入。v3 保留比較，不能以舊97.33%代表未見骨架效果。


In [ ]:
# 第 1 步：上傳新版完整 dataset.zip，內含 model/ 程式、data/ 四集合與 package-manifest.json。
# files.upload 回傳「檔名→上傳 bytes」字典；只允許一包，避免不知道正在使用哪個版本。
# BASE 使用目前 Colab 工作目錄加隨機名稱，重跑不會混到前一次資料，不沿用 C:\Users 等路徑。
# ZIP 內路徑先檢查是否為絕對路徑或含 ..；resolve 後也必須仍位於 BASE 內才能解壓。
# manifest 保存每個相對檔名的 SHA-256；解壓後重新算，確認程式與資料沒有混版或傳輸變動。
# 此核對不是模型成效驗證；只上傳 CSV 壓縮檔會缺必要程式與清單，不能直接取代完整套件。
# provenance 的 evaluation_protocol 必須是新版 skeleton-disjoint 規則。
# v2 的 420 筆候選是另一版資料，不能拿其證據來宣稱這批 v4 生成請求已經送過平台。

from pathlib import Path, PurePosixPath
import hashlib, json, zipfile, uuid
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("請只上傳一個新版訓練套件 ZIP")
BASE = Path.cwd() / ("msdss-" + uuid.uuid4().hex[:8])
BASE.mkdir()
with zipfile.ZipFile(next(iter(uploaded))) as archive:
    for entry in archive.infolist():
        name = PurePosixPath(entry.filename)
        if name.is_absolute() or '..' in name.parts or not (BASE / entry.filename).resolve().is_relative_to(BASE.resolve()):
            raise ValueError("ZIP 含不合法路徑")
    archive.extractall(BASE)
manifest = json.loads((BASE / 'package-manifest.json').read_text())
for name, expected in manifest.items():
    target = (BASE / name).resolve()
    if not target.is_relative_to(BASE.resolve()) or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
        raise ValueError("套件檔案不一致：" + name)
print("資料和程式位於", BASE)
provenance = json.loads((BASE / "data/provenance.json").read_text(encoding="utf-8"))
if provenance.get("evaluation_protocol") != "document-v4-skeleton-disjoint":
    raise ValueError("請上傳新版 v4 dataset.zip；舊420筆套件保留作基準。")


In [ ]:
# 第 2 步：用 sys.executable（目前 Notebook 的 Python）安裝 requirements 裡的固定依賴。
# 不直接呼叫可能指向另一個環境的 pip，也不搬 Windows 虛擬環境到 Colab。
# check_call 會等安裝結束，失敗就拋錯；此步不會開始訓練或使用 GPU。
# v5 新增的 HashingVectorizer 屬 sklearn，稍後不需要 fit 詞彙或下載語言模型。

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(BASE / 'model/requirements.txt')])

## 檢查資料
離線模式檢查平台、三類標籤、v3 輸入、配對與跨集合洩漏；不要求尚未存在的 HTTP 或 session 證據。


In [ ]:
# 第 3 步：檢查資料模式及跨集合關聯，這格沒有呼叫 model.fit，不會產生成績。
# MODE 控制來源條件；offline 允許生成資料，但仍查平台、三類標籤、pair_group 與 v3 原文。
# live 會要求目前版本每筆有真實 HTTP/session 證據；不是只改字串就完成實機驗證。
# 一般 test 可共享 train 的已見家族，模板／配對／來源仍要獨立；未知家族另放 challenge。
# READINESS 是 --check-only 的 JSON 報告，含 ready、splits、errors、warnings。
# returncode 非 0 就停止；顯示的獨立群組是相依關聯合併結果，不能當成真實購物 session 數。
# 通過只代表目前檢查條件成立，不保證 normal 的業務參數有效、標籤正確或模型能泛化。

MODE = "offline"  # 真實 HTTP 驗證改為 "live"
check = subprocess.run([sys.executable, "-X", "utf8", str(BASE / "model/train.py"),
    "--data-dir", str(BASE / "data"), "--evidence-root", str(BASE / "evidence"),
    "--mode", MODE, "--check-only"], capture_output=True, text=True, encoding="utf-8")
READINESS = json.loads(check.stdout) if check.stdout else {}
if check.returncode != 0:
    print(check.stdout or check.stderr)
    raise RuntimeError("資料檢查未通過，請處理 errors；尚未開始訓練。")

print("Data checks passed; mode:", READINESS.get("mode"))
for split, info in READINESS.get("splits", {}).items():
    print(f"{split}: {info['rows']} rows")
for warning in READINESS.get("warnings", []):
    print(warning)


## 訓練
預先固定樹數，不使用 test 或 Sylius validation 選參數。離線成績與平台實機驗證分開解讀。


In [ ]:
# 第 4 步：預先固定 TREES=150；只有 train 會傳給 fit，其他集合只做評估。
# MODE 沿用上一格，未執行上一格時仍預設 offline，不會默認已有 HTTP 證據。
# OUTPUT 的時間戳讓每次實驗有獨立目錄，避免新分數配到舊模型。
# command 是參數清單，路徑即使包含空白／中文也不用手動拼 shell 字串。
# run.stdout/stderr 分別保存正常與錯誤輸出；非 0 結束碼不視為訓練成功。
# v5 若有 12 個文件 header，x['train'] 為 450×2402；y['train'] 是 450 個 0/1/2 標籤。
# 2402 維＝舊統計 278＋4 通道×(512 雜湊桶＋11 符號＋8 正則)，不再是舊模型的 278 維。
# offline 產物標 exploratory=True、platform_http_verified=False，分數不代表平台漏洞或防護率。
# 輸出 inference/ 是模型部署套件，experiment.json 是來源／參數記錄，split.csv 可核對資料分工。

from datetime import datetime
MODE = globals().get("MODE", "offline")
TREES = 150
OUTPUT = BASE / "results" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
command = [sys.executable, "-X", "utf8", str(BASE / "model/train.py"),
    "--data-dir", str(BASE / "data"), "--evidence-root", str(BASE / "evidence"),
    "--mode", MODE, "--output", str(OUTPUT), "--trees", str(TREES)]
run = subprocess.run(command, capture_output=True, text=True, encoding="utf-8")
if run.stderr:
    print(run.stderr)
if run.returncode != 0:
    print(run.stdout)
    raise RuntimeError("訓練未完成，具體原因已顯示於上方。")

print("Training completed; the next cell shows test and validation results.")


## 三種保留評估分開呈現
一般 test、Sylius validation、保留語境群 challenge 不能混成同一分數。共同設定只顯示一次，完整指標與稽核資訊仍保存在下載檔案。


In [ ]:
# 第 5 步：同一個模型考 test、Sylius validation 與存在時的 challenge，不能混成一個成功率。
# REPORTS 依集合讀 report.json；keys 只選出要顯示的摘要，完整數據仍保存在檔案中。
# Accuracy=整體答對比例；Macro Precision/Recall/F1 先算各類再等權平均。
# 例如 100 筆正常中 5 筆被報為攻擊，normal_false_positive_rate 就是 0.05。
# SQLi 被判 XSS：三類分類算錯，但 attack_detection_recall 仍算有抓到攻擊。
# per_class 的 support 是該類真實筆數，Precision 的分母則是所有被預測成該類的筆數。
# shared 只將共同設定顯示一次；html.escape 避免 JSON 裡的字串被當成 HTML 執行。
# 獨立群組太少時不顯示信賴區間，不能把沒有區間解讀為沒有誤差。
# 已看過這些保留題的分數再反覆修改參數，就不再是完全未接觸的最終評估。

import json, html
from IPython.display import display, Markdown, HTML
EVALUATION_SPLITS = ['test', 'validation'] + (['challenge'] if (OUTPUT / 'challenge/report.json').is_file() else [])
REPORTS = {split: json.loads((OUTPUT / split / 'report.json').read_text(encoding='utf-8'))
           for split in EVALUATION_SPLITS}
keys = [('accuracy', 'Accuracy'), ('macro_precision', 'Macro Precision'),
        ('macro_recall', 'Macro Recall'), ('macro_f1', 'Macro-F1'),
        ('normal_false_positive_rate', 'Normal false-positive rate'),
        ('attack_detection_recall', 'Attack detection recall'), ('attack_miss_rate', 'Attack miss rate')]
titles = {'test': 'Skeleton-disjoint test', 'validation': 'Sylius validation', 'challenge': 'Held-out context challenge'}
lines = ['| Metric | ' + ' | '.join(titles[s] for s in EVALUATION_SPLITS) + ' |', '|---|' + '---:|' * len(EVALUATION_SPLITS)]
for key, title in keys:
    values = [REPORTS[split].get(key) for split in EVALUATION_SPLITS]
    rendered = [f'{value:.4f}' if value is not None else 'N/A' for value in values]
    lines.append('| ' + title + ' | ' + ' | '.join(rendered) + ' |')
display(Markdown('\n'.join(lines)))
lines = ['| Split | Class | Precision | Recall | F1 | N |', '|---|---|---:|---:|---:|---:|']
for split, report in REPORTS.items():
    for label in ('normal', 'sqli', 'xss'):
        r = report['per_class'][label]
        lines.append(f"| {split} | {label} | {r['precision']:.4f} | {r['recall']:.4f} | {r['f1-score']:.4f} | {int(r['support'])} |")
display(Markdown('\n'.join(lines)))
audit = REPORTS['test']['audit']
shared_keys = ('mode', 'exploratory', 'platform_http_verified', 'packages', 'feature_count',
               'training_seconds', 'counts', 'independent_groups', 'attack_skeleton_groups', 'evaluation_protocols', 'identical_feature_vectors_shared')
shared = {key: audit.get(key) for key in shared_keys}
display(HTML('<details><summary>Shared configuration and data checks (expand)</summary><pre>'
             + html.escape(json.dumps(shared, ensure_ascii=False, indent=2)) + '</pre></details>'))
for split, report in REPORTS.items():
    interval = report.get('confidence_interval', {})
    if not interval.get('available'):
        print(split + ': ' + interval.get('reason', 'No confidence interval available.'))
print('All detailed metrics and audits remain in report.json/report.md and CSV files.')


In [ ]:
# 圖表只讀既有 report/curves 數字，不重訓、不換門檻、不挑比較好看的樣本。
# 每個集合畫一張三面板圖：混淆矩陣（筆數＋逐列百分比）、ROC、PR。
# 矩陣列是真實、欄是預測；若 normal 那列為 [90, 6, 4]，就是正常 100 筆誤報 10 筆。
# fractions 同列為 [.90,.06,.04]，是除以該類真實總數，不是整張矩陣總數或各欄總數。
# ROC 逐類對其他類計算，x=FPR、y=TPR；灰虛線只是無辨別力排序的參考線。
# PR 的 x=Recall、y=Precision，圖例 AP 是 Average Precision，不是以梯形積分算出的 PR-AUC。
# AP 受該類比例影響；不同來源／比例的兩份資料不能只看 AP 大小斷言誰比較有效。
# evaluation_overview.png 存在各集合目錄；未安裝 Matplotlib 時原 CSV/JSON 數據仍可使用。

# One figure per split: matrix with count + row percentage, ROC, PR.
import csv
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('Matplotlib is unavailable; all numeric results remain in CSV/JSON files.')
else:
    for split in ['test', 'validation'] + (['challenge'] if (OUTPUT / 'challenge/report.json').is_file() else []):
        folder = OUTPUT / split
        report = json.loads((folder / 'report.json').read_text(encoding='utf-8'))
        labels = ['normal', 'sqli', 'xss']
        matrix = report['confusion_matrix']
        fractions = [[v / sum(row) if sum(row) else 0 for v in row] for row in matrix]
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
        ax = axes[0]
        image = ax.imshow(fractions, cmap='Blues', vmin=0, vmax=1)
        ax.set(xticks=range(3), yticks=range(3), xticklabels=labels, yticklabels=labels,
               xlabel='Predicted', ylabel='Actual', title='Count (row percentage)')
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f'{matrix[i][j]}\n({fractions[i][j]:.1%})', ha='center', va='center',
                        color='white' if fractions[i][j] > .5 else 'black')
        fig.colorbar(image, ax=ax, fraction=.046, pad=.04, label='Row proportion')
        with (folder / 'curves.csv').open(encoding='utf-8-sig', newline='') as handle:
            points = list(csv.DictReader(handle))
        for ax, curve in zip(axes[1:], ('ROC', 'PR')):
            for label in labels:
                values = [p for p in points if p['curve'] == curve and p['class'] == label]
                metric = report['one_vs_rest'][label].get('roc_auc' if curve == 'ROC' else 'average_precision')
                legend = f'{label} ({"AUC" if curve == "ROC" else "AP"}={metric:.3f})' if metric is not None else label
                xs, ys = [float(p['x']) for p in values], [float(p['y']) for p in values]
                if curve == 'PR':
                    ax.step(xs, ys, where='post', label=legend)
                else:
                    ax.plot(xs, ys, label=legend)
            if curve == 'ROC':
                ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
            ax.set(xlabel='FPR' if curve == 'ROC' else 'Recall',
                   ylabel='TPR' if curve == 'ROC' else 'Precision', title=curve,
                   xlim=(0, 1), ylim=(0, 1.02))
            ax.legend(fontsize=8)
        fig.suptitle(split + ' - ' + report['audit'].get('mode', 'unspecified'))
        fig.tight_layout()
        fig.savefig(folder / 'evaluation_overview.png', dpi=160)
        plt.show()
        plt.close(fig)


## 套件自我檢查與下載
`inference.zip` 內含模型、同版特徵／遮蔽／推論程式、依賴版本與自我檢查樣本。Windows、Ubuntu 各自解壓後都應先執行 `python predict.py --self-test`。Notebook 成功不等於已替你驗證所有作業系統。

In [ ]:
# 第 6 步：以套件自帶 train 探針核對預測一致性，不是再考 test/validation/challenge。
# self-test 比較三類機率，允許微小浮點誤差；搬到另一台 Windows/Linux 後也應再執行。
# 例如解壓 inference.zip 後，在該目錄執行：python predict.py --self-test --model .
# 模型、metadata、features、request_input、predict 與 manifest 必須同版一起搬移。
# 檔案 SHA 包含註解，不能只換新 .py 卻沿用舊套件清單，也不能把 v4 模型配 v5 特徵。
# results_download.zip 包含整次報告與 inference.zip；部署只需推論套件，不搬 Colab 環境。
# 目前 gateway 拒絕 exploratory 模型，離線產物不會因自測通過就變成正式閘道模型。

subprocess.check_call([sys.executable, str(OUTPUT / 'inference/predict.py'), '--self-test'])
import shutil
archive = shutil.make_archive(str(BASE / 'results_download'), 'zip', root_dir=OUTPUT)
files.download(archive)